# FYP Continual Learning Experiment

This notebook imports reusable logic from the Python scripts and runs the experiment interactively.

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "fyp_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from fyp_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from fyp_pipeline.experiment_runner import run_experiment, build_cl_summary, print_and_save_comparison_tables, generate_all_plots


## Vast.ai / GPU setup

On Vast.ai, upload the project folder to a location such as `/workspace/fyp`. Put the CSV files in `/workspace/fyp/data/processed`, then set `data_dir` below. For local runs, leave both values as `None`.

In [3]:
DATA_DIR = "/workspace/fyp_continuallearning_selfdistillation/data/processed"
OUTPUT_DIR = "/workspace/fyp_continuallearning_selfdistillation/outputs"

configure_vast_ai(DATA_DIR, OUTPUT_DIR, require_gpu=True)


Runtime diagnostics

Python         : 3.12.13

PyTorch        : 2.3.1+cu121

CUDA available : True

GPU            : NVIDIA GeForce RTX 3090

...

CPU cores      : 64

Vast.ai        : NO

Active device  : CUDA

Precision      : 32

{'paths': {'demand_csv': '/workspace/fyp_continuallearning_selfdistillation/data/processed/demand_forecasting.csv',
  'rl_csv': '/workspace/fyp_continuallearning_selfdistillation/data/processed/rl_environment.csv',
  'checkpoints': '/workspace/fyp_continuallearning_selfdistillation/outputs/checkpoints',
  'results': '/workspace/fyp_continuallearning_selfdistillation/outputs/results',
  'logs': '/workspace/fyp_continuallearning_selfdistillation/outputs/logs',
  'plots': '/workspace/fyp_continuallearning_selfdistillation/outputs/plots'},
 'tasks': [{'task_id': 1,
   'name': 'Baseline_2023_H1',
   'start': '2023-01-01',
   'end': '2023-05-31',
   'regime': 'baseline'},
  {'task_id': 2,
   'name': 'MegaSale_2023',
   'start': '2023-06-01',
   'end': '2023-12-31',
   'regime': 'mega_sale'},
  {'task_id': 3,
   'name': 'Baseline_2024_H1',
   'start': '2024-01-01',
   'end': '2024-05-31',
   'regime': 'baseline'},
  {'task_id': 4,
   'name': 'MegaSale_2024',
   'start': '2024-06-01',
   'end'

In [4]:
tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)


Loading datasets...

Demand CSV  : 9,864 rows × 66 cols

RL CSV      : 9,864 rows × 86 cols

✓ Data loaded and cleaned

┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Task ┃ Name             ┃ Period                   ┃ TFT rows ┃ RL rows ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ 1    │ Baseline_2023_H1 │ 2023-01-01 -> 2023-05-31 │ 1,359    │ 1,359   │
│ 2    │ MegaSale_2023    │ 2023-06-01 -> 2023-12-31 │ 1,926    │ 1,926   │
│ 3    │ Baseline_2024_H1 │ 2024-01-01 -> 2024-05-31 │ 1,368    │ 1,368   │
│ 4    │ MegaSale_2024    │ 2024-06-01 -> 2024-12-31 │ 1,926    │ 1,926   │
│ 5    │ Baseline_2025_H1 │ 2025-01-01 -> 2025-05-31 │ 1,359    │ 1,359   │
│ 6    │ MegaSale_2025    │ 2025-06-01 -> 2025-12-31 │ 1,926    │ 1,926   │
└──────┴──────────────────┴──────────────────────────┴──────────┴─────────┘

In [5]:
from fyp_pipeline.core_pipeline import CONFIG

CONFIG["hardware"]["compile"] = False
print(CONFIG["hardware"]["compile"])

False


In [ ]:
run_experiment(tft_tasks, rl_tasks)


==============================================================

  CONTINUAL LEARNING EXPERIMENT START

==============================================================

═══ MODEL TYPE: FORECASTING ═══

  ── CL Method: naive ──

...

Output()

ERROR: task 3 ewc: cannot reshape array of size 1228800 into shape (449,573,4)

Traceback (most recent call last):
  File "/workspace/fyp_continuallearning_selfdistillation/fyp_pipeline/experiment_runner.py", line 563, in run_experiment
    model, trainer = FORECAST_TRAINERS[cl_method](
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/fyp_continuallearning_selfdistillation/fyp_pipeline/experiment_runner.py", line 109, in train_forecast_ewc
    trainer.fit(model, train_loader, val_loader)
  File "/venv/main/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 584, in fit
    call._call_and_handle_interrupt(
  File "/venv/main/lib/python3.12/site-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/venv/main/lib/python3.12/site-packag

Task 4/6: MegaSale_2024

Output()

In [ ]:
cl_summary = build_cl_summary()
tables = print_and_save_comparison_tables(cl_summary)


In [ ]:
generate_all_plots(cl_summary)
